In [0]:
%pip install --quiet pdfplumber

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
%run ./fsr_config.py

In [0]:
import json, time, re, threading, collections
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from datetime import datetime, timezone

import requests
import urllib3
import pdfplumber
import pandas as pd

from pyspark.sql.functions import (
    col, when, trim, concat, lit, upper,
    regexp_replace, current_timestamp,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, IntegerType,
)

urllib3.disable_warnings()
METADATA_TABLE = f"{POC_CATALOG}.{POC_SCHEMA}.fsr_metadata_registry_test"
P1_MAX_PDFS = 10
LITELLM_API_KEY = "sk-cjRRha3Ejczz8AmnJKyQxA"

print("=== Process 1 — Metadata Extraction ===")
print(f"  Metadata table : {METADATA_TABLE}")
print(f"  Volumes        : {PDF_VOLUME_PATHS}")
print(f"  LLM model      : {LLM_MODEL}")
print(f"  Batch size     : {P1_BATCH_SIZE}")
print(f"  FORCE_RESET    : {FORCE_RESET}")
print(f"  LLM base URL   : {LITELLM_BASE_URL}")
print(f"  API key set    : {bool(LITELLM_API_KEY)}")

=== Process 1 — Metadata Extraction ===
  Metadata table : main.gp_services_sdg_poc.fsr_metadata_registry_test
  Volumes        : ['/Volumes/viud/ing_ud_fieldvision/fv_field_service_report', '/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/ecrt_reports']
  LLM model      : gemini-3-flash
  Batch size     : 4
  FORCE_RESET    : False
  LLM base URL   : https://dev-gateway.apps.gevernova.net
  API key set    : True


In [0]:
from pyspark.sql import SparkSession
from datetime import datetime, timezone
from pathlib import Path

spark = SparkSession.builder.getOrCreate()

# ── Create test metadata table if it doesn't exist ──────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {METADATA_TABLE} (
        {METADATA_TABLE_DDL_COLS}
    )
    USING DELTA
    COMMENT 'FSR metadata registry — TEST'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[OK] Table ready: {METADATA_TABLE}")

# ── Read watermark (skip if FORCE_RESET) ────────────────────────────────────
watermark_ms = 0
if not FORCE_RESET:
    row = spark.sql(f"SELECT MAX(ingested_at) AS wm FROM {METADATA_TABLE}").first()
    if row and row.wm:
        watermark_ms = int(row.wm.timestamp() * 1000)
        print(f"  Watermark: {row.wm} ({watermark_ms} ms)")
    else:
        print("  Watermark: None (first run — will scan all files)")
else:
    spark.sql(f"TRUNCATE TABLE {METADATA_TABLE}")
    print("  FORCE_RESET: table truncated, scanning all files")

# ── Scan volumes for new PDFs ───────────────────────────────────────────────
new_files = []
for vol in PDF_VOLUME_PATHS:
    print(f"\n  Scanning: {vol}")
    try:
        all_entries = dbutils.fs.ls(vol)
    except Exception as e:
        print(f"    [WARN] Cannot list volume: {e}")
        continue

    for f in all_entries:
        if f.path.endswith("/"):
            continue  # skip directories
        path = f.path.replace("dbfs:", "") if f.path.startswith("dbfs:") else f.path
        suffix = Path(path).suffix.lower()
        if suffix in SKIP_SUFFIXES:
            continue
        if f.modificationTime <= watermark_ms:
            continue
        new_files.append({
            "path": path,
            "name": f.name,
            "size": f.size,
            "mod_time": f.modificationTime,
            "source_volume": vol,
        })
    print(f"    Found {sum(1 for nf in new_files if nf['source_volume'] == vol)} new files")

# ── Cap at P1_MAX_PDFS ──────────────────────────────────────────────────────
if P1_MAX_PDFS and len(new_files) > P1_MAX_PDFS:
    new_files = new_files[:P1_MAX_PDFS]

print(f"\n[OK] {len(new_files)} files to process (cap={P1_MAX_PDFS})")
for nf in new_files[:5]:
    ts = datetime.fromtimestamp(nf['mod_time'] / 1000, tz=timezone.utc).isoformat()
    print(f"  {ts}  {nf['name'][:60]}  ({nf['size']} bytes)")
if len(new_files) > 5:
    print(f"  ... and {len(new_files) - 5} more")

[OK] Table ready: main.gp_services_sdg_poc.fsr_metadata_registry_test
  Watermark: None (first run — will scan all files)

  Scanning: /Volumes/viud/ing_ud_fieldvision/fv_field_service_report
    Found 15131 new files

  Scanning: /Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/ecrt_reports
    Found 2732 new files

[OK] 10 files to process (cap=10)
  2026-02-23T15:39:30+00:00  000496d8-fefa-4b6c-a654-791e328a4ddb  (47351998 bytes)
  2026-02-23T15:42:09+00:00  00063d90-f42d-45bb-933e-d941dc57a90a  (13684615 bytes)
  2026-02-23T15:40:32+00:00  000966e2-b88b-459c-8966-e2b88be59c9d  (13006929 bytes)
  2026-02-23T15:43:28+00:00  000969ff-716d-451d-8c49-fe08a82ef50a  (1849630 bytes)
  2026-02-23T15:39:50+00:00  00170e4e-fdec-45fa-a7e8-e87ada9711e8  (68052614 bytes)
  ... and 5 more


In [0]:
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, IntegerType,
)

# ── Build stub rows for the 10 discovered files ────────────────────────────
stub_rows = []
for nf in new_files:
    vol_path = nf["path"]
    doc_id = make_document_id(vol_path)
    pdf_name = Path(vol_path).stem  # GUID stem for FieldVision, filename for manual
    mod_ts = datetime.fromtimestamp(nf["mod_time"] / 1000, tz=timezone.utc)
    stub_rows.append({
        "document_id": doc_id,
        "volume_path": vol_path,
        "pdf_name": pdf_name,
        "source_volume": nf["source_volume"],
        "file_size_bytes": nf["size"],
        "file_last_modified": mod_ts,
        "metadata_status": MetadataStatus.PENDING,
        "chunk_status": ChunkStatus.PENDING,
        "metadata_retry_count": 0,
        "metadata_version": 1,
    })

schema = StructType([
    StructField("document_id", StringType(), False),
    StructField("volume_path", StringType(), False),
    StructField("pdf_name", StringType(), True),
    StructField("source_volume", StringType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("file_last_modified", TimestampType(), True),
    StructField("metadata_status", StringType(), False),
    StructField("chunk_status", StringType(), False),
    StructField("metadata_retry_count", IntegerType(), True),
    StructField("metadata_version", IntegerType(), True),
])

stub_df = spark.createDataFrame(stub_rows, schema=schema)
stub_df.createOrReplaceTempView("_fsr_stubs")

# ── MERGE stub rows into metadata table (per ADR-007) ──────────────────────
merge_sql = f"""
MERGE INTO {METADATA_TABLE} AS tgt
USING _fsr_stubs AS src
ON tgt.document_id = src.document_id
WHEN MATCHED THEN UPDATE SET
    tgt.metadata_status = '{MetadataStatus.PENDING}',
    tgt.chunk_status = '{ChunkStatus.PENDING}',
    tgt.file_last_modified = src.file_last_modified,
    tgt.file_size_bytes = src.file_size_bytes,
    tgt.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    document_id, volume_path, pdf_name, source_volume,
    file_size_bytes, file_last_modified,
    metadata_status, chunk_status,
    metadata_retry_count, metadata_version,
    ingested_at, updated_at
) VALUES (
    src.document_id, src.volume_path, src.pdf_name, src.source_volume,
    src.file_size_bytes, src.file_last_modified,
    src.metadata_status, src.chunk_status,
    src.metadata_retry_count, src.metadata_version,
    current_timestamp(), current_timestamp()
)
"""
result = spark.sql(merge_sql)
print(f"[OK] MERGE complete")

# ── Verify ──────────────────────────────────────────────────────────────────
count = spark.sql(f"SELECT COUNT(*) AS n FROM {METADATA_TABLE}").first().n
pending = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {METADATA_TABLE}
    WHERE metadata_status = '{MetadataStatus.PENDING}'
""").first().n
print(f"  Total rows: {count}, Pending: {pending}")
display(spark.sql(f"SELECT document_id, pdf_name, metadata_status, chunk_status, ingested_at FROM {METADATA_TABLE}"))

[OK] MERGE complete
  Total rows: 10, Pending: 10


document_id,pdf_name,metadata_status,chunk_status,ingested_at
e223d7ff12b06799,000969ff-716d-451d-8c49-fe08a82ef50a,pending,pending,2026-04-20T17:20:06.693Z
ce3b3e9ed1bc9fa9,00170e4e-fdec-45fa-a7e8-e87ada9711e8,pending,pending,2026-04-20T17:20:06.693Z
369111da7f69beb8,0022a975-bfd6-4540-a2a9-75bfd6254006,pending,pending,2026-04-20T17:20:06.693Z
df9dda62c7875352,002acea4-5f21-4784-aace-a45f21178466,pending,pending,2026-04-20T17:20:06.693Z
266e6fd93f857807,000496d8-fefa-4b6c-a654-791e328a4ddb,pending,pending,2026-04-20T17:20:06.693Z
5a69d842c0e57389,00063d90-f42d-45bb-933e-d941dc57a90a,pending,pending,2026-04-20T17:20:06.693Z
c0ea1aaaaf4addb4,001982f9-b830-448e-9326-9114c5451734,pending,pending,2026-04-20T17:20:06.693Z
4c21b0ee3f4a5923,000966e2-b88b-459c-8966-e2b88be59c9d,pending,pending,2026-04-20T17:20:06.693Z
2faf8b8116892f4e,001de483-07ac-4c2d-9de4-8307acfc2df7,pending,pending,2026-04-20T17:20:06.693Z
a0633f7cb795912f,00203008-c03c-4762-a030-08c03ce76258,pending,pending,2026-04-20T17:20:06.693Z


In [0]:
import pdfplumber

# ── Load pending rows ───────────────────────────────────────────────────────
pending_rows = spark.sql(f"""
    SELECT document_id, volume_path, pdf_name, metadata_retry_count
    FROM {METADATA_TABLE}
    WHERE metadata_status IN ('{MetadataStatus.PENDING}', '{MetadataStatus.FAILED}')
      AND (metadata_retry_count IS NULL OR metadata_retry_count < {P1_MAX_RETRIES})
""").collect()

print(f"[OK] {len(pending_rows)} documents to process")

# ── Extract first-page fields + page count from each PDF ───────────────────
extractions = {}  # document_id -> {title, llm_fields, page_count}
failures = {}     # document_id -> error_msg

for row in pending_rows:
    doc_id = row.document_id
    vol_path = row.volume_path
    try:
        with pdfplumber.open(vol_path) as pdf:
            page_count = len(pdf.pages)
            first_page = pdf.pages[0]
            text = first_page.extract_text() or ""

            lines = text.splitlines()

            # Title = lines before the first colon-delimited field
            title_lines = []
            for line in lines:
                if ":" in line:
                    break
                title_lines.append(line.strip())
            title = " ".join(title_lines).strip() or None

            # Key:value pairs for LLM normalization
            llm_fields = {"PDF Name / path / identifier": vol_path}
            current_key = None
            for line in lines:
                if ":" in line:
                    parts = line.split(":", 1)
                    key = parts[0].strip()
                    value = parts[1].strip()
                    current_key = key
                    if key in llm_fields:
                        llm_fields[key] = f"{llm_fields[key]} | {value}"
                    else:
                        llm_fields[key] = value
                elif current_key:
                    llm_fields[current_key] += " " + line.strip()

            extractions[doc_id] = {
                "title": title,
                "llm_fields": llm_fields,
                "page_count": page_count,
            }
            print(f"  [OK] {row.pdf_name[:40]}  pages={page_count}  fields={len(llm_fields)}")

    except Exception as e:
        failures[doc_id] = str(e)[:500]
        print(f"  [FAIL] {row.pdf_name[:40]}: {e}")

print(f"\n[OK] Extracted: {len(extractions)}, Failed: {len(failures)}")

# Quick peek at one extraction
if extractions:
    sample_id = next(iter(extractions))
    sample = extractions[sample_id]
    print(f"\n--- Sample ({sample_id}) ---")
    print(f"  Title: {sample['title']}")
    print(f"  Page count: {sample['page_count']}")
    print(f"  Fields ({len(sample['llm_fields'])}):")
    for k, v in list(sample['llm_fields'].items())[:8]:
        print(f"    {k}: {str(v)[:80]}")

[OK] 10 documents to process
  [OK] 000969ff-716d-451d-8c49-fe08a82ef50a  pages=17  fields=5
  [OK] 00170e4e-fdec-45fa-a7e8-e87ada9711e8  pages=611  fields=5
  [OK] 0022a975-bfd6-4540-a2a9-75bfd6254006  pages=205  fields=5
  [OK] 002acea4-5f21-4784-aace-a45f21178466  pages=225  fields=4
  [OK] 000496d8-fefa-4b6c-a654-791e328a4ddb  pages=204  fields=6
  [OK] 00063d90-f42d-45bb-933e-d941dc57a90a  pages=51  fields=5
  [OK] 001982f9-b830-448e-9326-9114c5451734  pages=229  fields=6
  [OK] 000966e2-b88b-459c-8966-e2b88be59c9d  pages=60  fields=4
  [OK] 001de483-07ac-4c2d-9de4-8307acfc2df7  pages=24  fields=5
  [OK] 00203008-c03c-4762-a030-08c03ce76258  pages=33  fields=5

[OK] Extracted: 10, Failed: 0

--- Sample (e223d7ff12b06799) ---
  Title: GE Power Power Services Control Service Report KASHIMA
  Page count: 17
  Fields (5):
    PDF Name / path / identifier: /Volumes/viud/ing_ud_fieldvision/fv_field_service_report/000969ff-716d-451d-8c49
    Outage Start Date: 27 Aug 2023
    ESN/SY: 299

In [0]:
import json, re, requests, time

# ── LLM prompts (from DS reference pipeline) ───────────────────────────────

SYSTEM_PROMPT = (
    "You are an expert in structuring technical data. "
    "Your task is to process the provided input json and output a structured/normalized "
    "JSON format according to user instructions. Focus on clarity, completeness, and "
    "following the JSON schema provided. Do not include any internal reasoning or system "
    "details in the output. Ensure all responses are fact-based, and safe. "
    "Follow responsible AI principles without over-restricting harmless tasks."
)

NORMALIZATION_PROMPT_SUFFIX = """Your task is to process JSON input and output a normalized JSON format
with the following fields only:

ESN, Equipment Sys ID, Equipment Type, Equipment Class / Code, Event Type,
EV Project ID, EV Equipment Event ID, OFS Event ID, FSP project ID, xxx project id, PDF Name / path / identifier,
FSR Number (#), Report Issued Date, Outage Start Date, Outage End Date.

If the PDF field contains an Oracle Project Id, map it to OFS Event ID only. But do not include field values that start with "EV" and "EVP" under OFS Event ID.
Map the PDF field containing "EV-" to EV Equipment Event ID only.
Map the PDF field containing "EVP-" to EV Project ID only.
Map the PDF field containing "SY" to Equipment Sys ID only.
If the PDF field contains a Field Service Project Id or "FSP-", map it to FSP project ID only.
If the PDF field contains a project id that starts with "XXX" (i.e. A-, C-, etc.), map it to xxx project id only.

**Strictly adhere to the following instructions while extracting and normalizing data**:
Do not miss any information that is present in the input and try to be as much accurate as possible in retrieving the values for the above fields.
**When extracting data, if a record contains multiple distinct values across ESN and Equipment Sys ID fields, split the record into separate rows by pairing values positionally (first with first, second with second, etc.), while duplicating all other field values unchanged (except for Equipment Type and Equipment Class / Code). Set the Equipment Type and Equipment Class / Code field values to empty strings in the split rows. Do not split or omit any parts for other field values even if multiple distinct values are present.**
If no exact match is found for a field, see if you can infer it from similar labels or context.
If no relevant information is found, output it as an empty string.
Consider as many records as provided in the input batch. Do not omit any records.
Do not include any reasoning or commentary, only valid JSON output.
All dates must be normalized to YYYY-MM-DD format.

For the field Event Type, only use values from the following allowed list:
Training Cost Accumulation, Unusual, Training Open Enrollment - Costs, TX Repairs,
Major Inspection (Field Rewind), Major Inspection (MI), Tooling(GE), C Inspection,
null, Training On Site Training, A Inspection, Services Warranty,
Borescope Inspection (BI), Major Inspection (Robotic), Performance Testing,
Upgrade - PMO Billing only, Digital, Non CSA-MMP Billing,
Hot Gas Path Inspection (HGPI), Initial Spares, Combustion Inspection (CI),
Training Open Enrollment - Billing, Stand Alone Small Upgrade, Large Call-Out,
On Site Services, Call-Out, TX Parts, B Inspection, Training Simulation,
Post COD New Unit Warranty, Major Inspection (Rotor Out), Stand Alone Large Upgrade,
OP Spares, Remote Diagnostics, Minor Inspection, Onsite Services SP,
Major Inspection (Stator Rewind)
"""


def _strip_json_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def call_llm(prompt):
    base = LITELLM_BASE_URL.rstrip("/")
    urls = [f"{base}/chat/completions", f"{base}/v1/chat/completions"]
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    }
    headers = {
        "Authorization": f"Bearer {LITELLM_API_KEY}",
        "Content-Type": "application/json",
    }
    for attempt in range(1, 4):
        for url in urls:
            try:
                resp = requests.post(url, headers=headers, json=payload,
                                     timeout=120, verify=LLM_VERIFY_SSL)
                if resp.status_code == 404:
                    continue
                resp.raise_for_status()
                return resp.json()["choices"][0]["message"]["content"]
            except Exception as e:
                if "CERTIFICATE_VERIFY_FAILED" in str(e):
                    continue
                if attempt == 3:
                    raise
                wait = 5 * (2 ** (attempt - 1))
                print(f"    [WARN] LLM attempt {attempt} failed ({e}); retrying in {wait}s")
                time.sleep(wait)
    raise RuntimeError("LLM call failed after all retries")


# ── Send extractions to LLM in batches ──────────────────────────────────────
COLUMN_MAP = {
    "ESN": "esn",
    "Equipment Sys ID": "equipment_sys_id",
    "Equipment Type": "equipment_type",
    "Equipment Class / Code": "equipment_code",
    "Event Type": "event_type",
    "EV Project ID": "ev_project_id",
    "EV Equipment Event ID": "ev_equipment_event_id",
    "OFS Event ID": "ofs_event_id",
    "FSP project ID": "fsp_project_id",
    "xxx project id": "xxx_project_id",
    "PDF Name / path / identifier": "_volume_path",
    "FSR Number (#)": "fsr_number",
    "Report Issued Date": "report_issued_date",
    "Outage Start Date": "outage_start_date",
    "Outage End Date": "outage_end_date",
}

doc_ids_ordered = list(extractions.keys())
all_llm_fields = [extractions[d]["llm_fields"] for d in doc_ids_ordered]

# Batch into groups of P1_BATCH_SIZE
normalized_by_doc = {}  # document_id -> normalized dict
llm_failures = {}       # document_id -> error

for i in range(0, len(all_llm_fields), P1_BATCH_SIZE):
    batch_fields = all_llm_fields[i:i + P1_BATCH_SIZE]
    batch_doc_ids = doc_ids_ordered[i:i + P1_BATCH_SIZE]
    batch_num = (i // P1_BATCH_SIZE) + 1

    prompt = (
        f"Here is a JSON list of PDF fields:\n{json.dumps(batch_fields, indent=2)}\n\n"
        + NORMALIZATION_PROMPT_SUFFIX
    )

    try:
        print(f"  [B{batch_num}] Sending {len(batch_fields)} records to LLM...")
        raw = call_llm(prompt)
        cleaned = _strip_json_fences(raw)
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict) and "FSR_data" in parsed:
            parsed = parsed["FSR_data"]
        if not isinstance(parsed, list):
            parsed = [parsed]
        print(f"  [B{batch_num}] Got {len(parsed)} normalized rows")

        # Map LLM rows back to document_ids via volume_path
        path_to_doc = {}
        for did in batch_doc_ids:
            vol_path = extractions[did]["llm_fields"]["PDF Name / path / identifier"]
            path_to_doc[vol_path] = did

        for row in parsed:
            row_mapped = {COLUMN_MAP.get(k, k): v for k, v in row.items() if k in COLUMN_MAP}
            vol_path = row_mapped.pop("_volume_path", "")
            doc_id = path_to_doc.get(vol_path)
            if doc_id and doc_id not in normalized_by_doc:
                normalized_by_doc[doc_id] = row_mapped

        # Check for any docs that didn't get a normalized row
        for did in batch_doc_ids:
            if did not in normalized_by_doc and did not in llm_failures:
                llm_failures[did] = "LLM returned no matching row for this document"

    except Exception as e:
        print(f"  [B{batch_num}] LLM FAILED: {e}")
        for did in batch_doc_ids:
            llm_failures[did] = str(e)[:500]

print(f"\n[OK] Normalized: {len(normalized_by_doc)}, LLM failures: {len(llm_failures)}")

# Show a sample
if normalized_by_doc:
    sample_id = next(iter(normalized_by_doc))
    print(f"\n--- Sample ({sample_id}) ---")
    for k, v in normalized_by_doc[sample_id].items():
        print(f"  {k}: {v}")

  [B1] Sending 4 records to LLM...
  [B1] Got 5 normalized rows
  [B2] Sending 4 records to LLM...
  [B2] Got 4 normalized rows
  [B3] Sending 2 records to LLM...
  [B3] Got 3 normalized rows

[OK] Normalized: 10, LLM failures: 0

--- Sample (e223d7ff12b06799) ---
  esn: 299129
  equipment_sys_id: SY0502614
  equipment_type: 
  equipment_code: 
  event_type: 
  ev_project_id: EVP-536430
  ev_equipment_event_id: EV-156709
  ofs_event_id: 
  fsp_project_id: 
  xxx_project_id: PMX-EG0-005986
  fsr_number: 
  report_issued_date: 2023-11-27
  outage_start_date: 2023-08-27
  outage_end_date: 


In [0]:
from pyspark.sql.functions import col, when, trim, concat, lit, upper, regexp_replace, current_timestamp
from pyspark.sql.types import StringType
import pandas as pd

# ── Load IBAT and Event Vision lookup tables ────────────────────────────────
enrichment_enabled = True
ibat_df = None
ev_sot_df = None

try:
    ibat_df = spark.read.table(IBAT_EQUIPMENT_TABLE).select(
        upper(trim(col("equipment_sys_id"))).cast(StringType()).alias("ibat_equipment_sys_id"),
        upper(trim(col("equip_serial_number"))).cast(StringType()).alias("ibat_equip_serial_number"),
        col("equipment_type").cast(StringType()).alias("ibat_equipment_type"),
        col("equipment_sub_class").cast(StringType()).alias("ibat_equipment_code"),
    )
    ibat_df.limit(1).count()
    print(f"[OK] IBAT loaded: {IBAT_EQUIPMENT_TABLE}")
except Exception as e:
    print(f"[WARN] IBAT not accessible ({e}). Continuing without IBAT enrichment.")
    enrichment_enabled = False

try:
    ev_sot_df = spark.read.table(EVENT_VISION_SOT_TABLE).select(
        col("ev_project_id").cast(StringType()).alias("sot_ev_project_id"),
        col("ev_equipment_event_id").cast(StringType()).alias("sot_ev_equipment_event_id"),
        col("ev_gtm_id").cast(StringType()).alias("sot_ev_gtm_id"),
        col("fsp_project_id").cast(StringType()).alias("sot_fsp_project_id"),
        col("ev_event_type").cast(StringType()).alias("sot_event_type"),
        col("p6_outage_start_date").cast(StringType()).alias("sot_outage_start_date"),
        col("p6_outage_end_date").cast(StringType()).alias("sot_outage_end_date"),
    )
    ev_sot_df.limit(1).count()
    print(f"[OK] Event Vision loaded: {EVENT_VISION_SOT_TABLE}")
except Exception as e:
    print(f"[WARN] Event Vision not accessible ({e}). Continuing without EV enrichment.")
    if not ibat_df:
        enrichment_enabled = False

# ── Build success records with enrichment ───────────────────────────────────
success_records = []

for doc_id, norm in normalized_by_doc.items():
    ext = extractions.get(doc_id, {})
    rec = {
        "document_id": doc_id,
        "title": ext.get("title"),
        "page_count": ext.get("page_count"),
        "esn": norm.get("esn", ""),
        "equipment_sys_id": norm.get("equipment_sys_id", ""),
        "equipment_type": norm.get("equipment_type", ""),
        "equipment_code": norm.get("equipment_code", ""),
        "event_type": norm.get("event_type", ""),
        "ev_project_id": norm.get("ev_project_id", ""),
        "ev_equipment_event_id": norm.get("ev_equipment_event_id", ""),
        "ofs_event_id": norm.get("ofs_event_id", ""),
        "fsp_project_id": norm.get("fsp_project_id", ""),
        "xxx_project_id": norm.get("xxx_project_id", ""),
        "fsr_number": norm.get("fsr_number", ""),
        "report_issued_date": norm.get("report_issued_date", ""),
        "outage_start_date": norm.get("outage_start_date", ""),
        "outage_end_date": norm.get("outage_end_date", ""),
        "esn_source": "llm" if norm.get("esn") else None,
    }
    success_records.append(rec)

# ── Enrich with IBAT + EV if available ──────────────────────────────────────
if enrichment_enabled and ibat_df:
    pdf = pd.DataFrame(success_records)
    fsr_df = spark.createDataFrame(pdf)
    fsr_df = fsr_df.select([col(c).cast(StringType()).alias(c) if c != "page_count"
                            else col(c) for c in fsr_df.columns])

    # Save pre-enrichment ESN to track source
    fsr_df = fsr_df.withColumn("_pre_ibat_esn", col("esn"))

    # Strip prefixes for join
    fsr_df = (fsr_df
        .withColumn("fsp_project_id_stripped", regexp_replace(col("fsp_project_id"), "FSP-", ""))
        .withColumn("ev_project_id_stripped", regexp_replace(col("ev_project_id"), "EVP-", ""))
        .withColumn("ev_equipment_event_id_stripped", regexp_replace(col("ev_equipment_event_id"), "EV-", ""))
    )

    # IBAT join
    enriched = fsr_df.join(
        ibat_df,
        (upper(trim(fsr_df["equipment_sys_id"])) == ibat_df["ibat_equipment_sys_id"])
        | (upper(trim(fsr_df["esn"])) == ibat_df["ibat_equip_serial_number"]),
        "left",
    )
    enriched = (enriched
        .withColumn("esn", when((col("esn").isNull()) | (col("esn") == ""), col("ibat_equip_serial_number")).otherwise(col("esn")))
        .withColumn("equipment_sys_id", when((col("equipment_sys_id").isNull()) | (col("equipment_sys_id") == ""), col("ibat_equipment_sys_id")).otherwise(col("equipment_sys_id")))
        .withColumn("equipment_type", when((col("equipment_type").isNull()) | (col("equipment_type") == ""), col("ibat_equipment_type")).otherwise(col("equipment_type")))
        .withColumn("equipment_code", when((col("equipment_code").isNull()) | (col("equipment_code") == ""), col("ibat_equipment_code")).otherwise(col("equipment_code")))
    )

    # EV join
    if ev_sot_df:
        enriched = enriched.join(
            ev_sot_df,
            (enriched["ev_project_id_stripped"] == ev_sot_df["sot_ev_project_id"])
            | (enriched["ev_equipment_event_id_stripped"] == ev_sot_df["sot_ev_equipment_event_id"])
            | (enriched["ofs_event_id"] == ev_sot_df["sot_ev_gtm_id"])
            | (enriched["fsp_project_id_stripped"] == ev_sot_df["sot_fsp_project_id"]),
            "left",
        )
        enriched = (enriched
            .withColumn("event_type", when((col("event_type").isNull()) | (col("event_type") == ""), col("sot_event_type")).otherwise(col("event_type")))
            .withColumn("outage_start_date", when((col("outage_start_date").isNull()) | (col("outage_start_date") == ""), col("sot_outage_start_date")).otherwise(col("outage_start_date")))
            .withColumn("outage_end_date", when((col("outage_end_date").isNull()) | (col("outage_end_date") == ""), col("sot_outage_end_date")).otherwise(col("outage_end_date")))
        )

    # ESN source
    enriched = enriched.withColumn("esn_source",
        when(col("_pre_ibat_esn").isNotNull() & (col("_pre_ibat_esn") != ""), lit("llm"))
        .when(col("esn").isNotNull() & (col("esn") != ""), lit("ibat"))
        .otherwise(lit(None))
    )

    # Dedup by document_id (IBAT join can multiply rows)
    enriched = enriched.dropDuplicates(["document_id"])

    # Collect back
    keep_cols = ["document_id", "title", "page_count", "esn", "esn_source",
                 "equipment_sys_id", "equipment_type", "equipment_code",
                 "event_type", "ev_project_id", "ev_equipment_event_id",
                 "ofs_event_id", "fsp_project_id", "xxx_project_id",
                 "fsr_number", "report_issued_date", "outage_start_date", "outage_end_date"]
    result_pdf = enriched.select(keep_cols).toPandas()
    success_records = result_pdf.where(result_pdf.notna(), None).to_dict("records")
    print(f"[OK] Enrichment complete: {len(success_records)} records")
else:
    print("[INFO] Skipping enrichment — using LLM-only results")

# ── MERGE results into metadata table ───────────────────────────────────────
update_rows = []
for rec in success_records:
    update_rows.append(rec)

# Also handle extraction-level failures
for doc_id, err in {**failures, **llm_failures}.items():
    update_rows.append({
        "document_id": doc_id,
        "_is_failure": True,
        "_error": err,
    })

# Build success DataFrame and merge
success_only = [r for r in update_rows if not r.get("_is_failure")]
if success_only:
    sdf = spark.createDataFrame(pd.DataFrame(success_only))
    sdf.createOrReplaceTempView("_fsr_success")

    spark.sql(f"""
        MERGE INTO {METADATA_TABLE} AS tgt
        USING _fsr_success AS src
        ON tgt.document_id = src.document_id
        WHEN MATCHED THEN UPDATE SET
            tgt.title = src.title,
            tgt.esn = src.esn,
            tgt.esn_source = src.esn_source,
            tgt.equipment_sys_id = src.equipment_sys_id,
            tgt.equipment_type = src.equipment_type,
            tgt.equipment_code = src.equipment_code,
            tgt.event_type = src.event_type,
            tgt.ev_project_id = src.ev_project_id,
            tgt.ev_equipment_event_id = src.ev_equipment_event_id,
            tgt.ofs_event_id = src.ofs_event_id,
            tgt.fsp_project_id = src.fsp_project_id,
            tgt.xxx_project_id = src.xxx_project_id,
            tgt.fsr_number = src.fsr_number,
            tgt.report_issued_date = src.report_issued_date,
            tgt.outage_start_date = src.outage_start_date,
            tgt.outage_end_date = src.outage_end_date,
            tgt.page_count = src.page_count,
            tgt.metadata_status = 'ok',
            tgt.metadata_error = NULL,
            tgt.updated_at = current_timestamp()
    """)
    print(f"[OK] Updated {len(success_only)} rows as metadata_status=ok")

# Handle failures
fail_only = [r for r in update_rows if r.get("_is_failure")]
for f in fail_only:
    spark.sql(f"""
        UPDATE {METADATA_TABLE}
        SET metadata_status = CASE
                WHEN metadata_retry_count >= {P1_MAX_RETRIES - 1} THEN '{MetadataStatus.SKIPPED}'
                ELSE '{MetadataStatus.FAILED}'
            END,
            metadata_error = '{f["_error"][:200].replace("'", "''")}',
            metadata_retry_count = COALESCE(metadata_retry_count, 0) + 1,
            updated_at = current_timestamp()
        WHERE document_id = '{f["document_id"]}'
    """)
if fail_only:
    print(f"[OK] Updated {len(fail_only)} rows as failed/skipped")

# ── Final verification ──────────────────────────────────────────────────────
print("\n--- Final table state ---")
display(spark.sql(f"""
    SELECT document_id, pdf_name, title, esn, esn_source, equipment_type,
           event_type, report_issued_date, metadata_status, page_count
    FROM {METADATA_TABLE}
"""))

[OK] IBAT loaded: vgpd.prm_std_views.ibat_equipment_mst
[OK] Event Vision loaded: vgpd.fsr_std_views.eventmgmt_event_vision_sot
[OK] Enrichment complete: 10 records
[OK] Updated 10 rows as metadata_status=ok

--- Final table state ---


document_id,pdf_name,title,esn,esn_source,equipment_type,event_type,report_issued_date,metadata_status,page_count
a0633f7cb795912f,00203008-c03c-4762-a030-08c03ce76258,Report Forced outage POMI unit 7 Work Scope PAITON POWER PLANT,null,null,Boiler,null,,ok,33
2faf8b8116892f4e,001de483-07ac-4c2d-9de4-8307acfc2df7,"GE Power Power Services Intervention on pump CV PP0006 doel 4 Due to high vibrations , checking pump coupling and alignments DOEL",null,null,Auxiliaries,Call-Out,2021-06-29,ok,24
4c21b0ee3f4a5923,000966e2-b88b-459c-8966-e2b88be59c9d,LES ROKEBY UNIT 2 A INSPECTION 2021 A inspection ROKEBY STATION,804605,llm,Gas Turbine,A Inspection,,ok,60
c0ea1aaaaf4addb4,001982f9-b830-448e-9326-9114c5451734,F KIRINA GT-11 A Inspection F KIRINA,G00277,llm,Gas Turbine,A Inspection,2024-11-03,ok,229
5a69d842c0e57389,00063d90-f42d-45bb-933e-d941dc57a90a,Unit 14 visual A Inspection MIDLAND COGEN,810893,llm,Gas Turbine,Hot Gas Path Inspection (HGPI),2025-10-20,ok,51
266e6fd93f857807,000496d8-fefa-4b6c-a654-791e328a4ddb,Anta GT02 C-Inspection Gas turbine C-Inspection ANTA,G00066,llm,Gas Turbine,C Inspection,2025-01-03,ok,204
df9dda62c7875352,002acea4-5f21-4784-aace-a45f21178466,ENGIE DROGENBOS TV 23 Generator and Seal Oil system major inspection ( rotor out) DROGENBOS,null,null,Generator,Major Inspection (MI),,ok,225
369111da7f69beb8,0022a975-bfd6-4540-a2a9-75bfd6254006,Installation AGP et Implémentation de la FMI 7584G1 AGP NAAMA,299309,llm,Gas Turbine,Hot Gas Path Inspection (HGPI),,ok,205
ce3b3e9ed1bc9fa9,00170e4e-fdec-45fa-a7e8-e87ada9711e8,GE Power Power Services Semi Annual Borescope Inspections Borescope and NDE Inspections with Additional Scope OLEANDER,297838,llm,Gas Turbine,Hot Gas Path Inspection (HGPI),2023-11-06,ok,611
e223d7ff12b06799,000969ff-716d-451d-8c49-fe08a82ef50a,GE Power Power Services Control Service Report KASHIMA,299129,llm,Gas Turbine,Non CSA-MMP Billing,2023-11-27,ok,17
